In [67]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import re
import time

In [ ]:
category=input('Category=')
category

In [120]:
# Initialize WebDriver
driver = webdriver.Chrome()
driver.maximize_window()

medicine_data = []  # Store scraped data


file_name = category
file_name = file_name.strip() + ".xlsx" 
directory = "D:/scrap/"
full_path = directory + file_name 

base_url = "https://medeasy.health/"
url = base_url + category

# Get total pages dynamically
driver.get(url)
time.sleep(3)

pagination_text = driver.find_element(By.XPATH, "//div[@class='pagination-total-text flex items-center']//p").text
total_pages = int(re.search(r'\((\d+) Pages\)', pagination_text).group(1))  # Extract total page count
total_products = int(re.search(r'of (\d+)', pagination_text).group(1))

print(f"Total Pages Detected: {total_pages}")
print(f"Total Products Found: {total_products}")

page = 223
while page <= total_pages:
    driver.get(f"{url}?page={page}")
    time.sleep(2)  # Allow page to load properly

    WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, "//*[@id='CategoryGrid']//a")))

    # Determine iteration count based on page number
    if page < total_pages:
        product_count = 10  # First few pages always have 10 products
    else:
        product_count = total_products % 10  # Last page has remaining products

    print(f"Scraping {product_count} products from page {page}...")

    for i in range(1, product_count + 1):  # Iterate from 1 to product_count
        product_xpath = f"//*[@id='CategoryGrid']//a[{i}]"  # Select product dynamically

        try:
            product = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.XPATH, product_xpath))
            )
            product.click()
        except Exception as e:
            print(f"Skipping product {i} due to error: {e}")
            continue

        time.sleep(2)

        try:
            try:
                product_name = driver.find_element(By.XPATH, "//h1[contains(@class, 'text-brand-dark')]").text
            except:
                product_name = ""
            try:
                dosage = driver.find_element(By.XPATH, "//h1/span[contains(@class, 'text-brand-gray')]").text
            except:
                dosage = ""
            try:
                types = driver.find_element(By.XPATH, "//h3[contains(@class, 'text-brand-gray')][1]").text
            except:
                types = ""
            try:
                medicine = driver.find_element(By.XPATH, "//h3[contains(@class, 'generic-text-color')]").text
            except:
                medicine = ""
            try:
                company = driver.find_element(By.XPATH, "//h3[contains(@class, 'text-brand-gray')][2]").text
            except:
                company = ""
            try:
                image_url = driver.find_element(By.XPATH, "//img[@alt='Product gallery']").get_attribute("src")
            except:
                image_url = ""
            try:
                price_text = driver.find_element(By.XPATH, "//del[contains(text(), 'MRP')]").text
            except:
                price_text = ""
            try:
                product_url = driver.find_element(By.XPATH,"//*[@rel='canonical']").get_attribute('href')
            except:
                product_url = ""
            try:
                strip_type = driver.find_element(By.XPATH,"//*[@class='flex-1 text-md text-brand-dark text-center']").text
            except:
                strip_type = ""
            try:
                stock =  driver.find_element(By.XPATH,"//span[@class='text-[11px] md:text-xs font-bold text-brand-danger uppercase inline-block rounded-lg py-2']").text
            except:
                stock = "In Stock"
            
            price = price_text.split("৳")[1].strip()
            name_only = product_name.replace(dosage, "").strip()
            full_name = f"{name_only} {dosage}"

            medicine_data.append({
                "Name": full_name,
                "Type": types,
                "Medicine": medicine,
                "Company": company,
                "MRP": price,
                "Strip": strip_type,
                "image_url": image_url,
                "product_link": product_url,
                "Stock": stock
            })

        except Exception as e:
            print(f"Error while scraping product: {e}")

        driver.back()  
        time.sleep(2)

    if page % 1 == 0:
        df = pd.DataFrame(medicine_data)
        backup_file = f"{directory}{category.strip()}_backup.xlsx"
        df.to_excel(backup_file, index=False, engine="openpyxl")
        print(f"✅ Backup saved at page {page}: {backup_file}")
    
    page += 1  


df = pd.DataFrame(medicine_data)
print(df)


df.to_excel(full_path, index=False, engine='openpyxl')
print(f"File saved successfully at {full_path}")

driver.quit()

Total Pages Detected: 292
Total Products Found: 2913
Scraping 10 products from page 223...
✅ Backup saved at page 223: D:/scrap/otc-medicine_backup.xlsx
Scraping 10 products from page 224...
✅ Backup saved at page 224: D:/scrap/otc-medicine_backup.xlsx
Scraping 10 products from page 225...
✅ Backup saved at page 225: D:/scrap/otc-medicine_backup.xlsx
Scraping 10 products from page 226...
✅ Backup saved at page 226: D:/scrap/otc-medicine_backup.xlsx
Scraping 10 products from page 227...
✅ Backup saved at page 227: D:/scrap/otc-medicine_backup.xlsx
Scraping 10 products from page 228...
✅ Backup saved at page 228: D:/scrap/otc-medicine_backup.xlsx
Scraping 10 products from page 229...
✅ Backup saved at page 229: D:/scrap/otc-medicine_backup.xlsx
Scraping 10 products from page 230...
✅ Backup saved at page 230: D:/scrap/otc-medicine_backup.xlsx
Scraping 10 products from page 231...
✅ Backup saved at page 231: D:/scrap/otc-medicine_backup.xlsx
Scraping 10 products from page 232...
✅ Backup 